# VideoRSI · One-Pass T4 Demo (Wan2.1 → SeedVR2)

**Goal:** Make **one** video, upscale it with **actual SeedVR2**, and export both MP4s.

```
Short user prompt → (optional OpenAI-compatible API expansion)
                  → Wan2.1 T2V 1.3B → source.mp4
                  → SeedVR2 3B FP8   → enhanced.mp4
```

**No loops, retries, LoRA, or training in this prototype.**

### Important limitations
- Designed for a **single Google Colab Tesla T4 (~15–16 GB VRAM)**; enable **Runtime → Change runtime type → T4 GPU**.
- We use **Wan2.1 1.3B**, not Wan2.2 5B. The latter's documented minimum is approximately 24 GB in official inference settings.
- This notebook has been **statically checked**, but **has not been executed on a real Colab T4 here**; actual GPU runtime, system RAM, free disk and package changes can cause failures.
- **SeedVR2 upscaling is non-optional**: if SeedVR2 fails, the notebook raises an error rather than silently replacing it with bicubic/ESRGAN.
- Model downloads can require **many GB of free disk**; this is best run in a fresh Colab session.

**References:** [Wan2.1 official](https://github.com/Wan-Video/Wan2.1), [Wan2.1 Diffusers weights](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B-Diffusers), [SeedVR2 CLI (T4 support)](https://github.com/delvilo/SeedVR2-CLI).

## 1. Confirm GPU and configure the demo

In [4]:
import os, sys, shutil, subprocess, json, pathlib, textwrap
import torch

assert torch.cuda.is_available(), "Please switch Colab to a T4 GPU runtime first."
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('Python:', sys.version.split()[0], '| Torch:', torch.__version__)
print('Disk free (GiB):', round(shutil.disk_usage('/content').free / 2**30, 1))
if shutil.disk_usage('/content').free < 22 * 2**30:
    print('WARNING: low disk space. Weight downloads may fail. Consider freeing disk.')

ROOT = pathlib.Path('/content/video_onepass_demo')
ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_MP4 = ROOT / 'source_wan.mp4'
UPSCALED_MP4 = ROOT / 'enhanced_seedvr2.mp4'

# ===== Change these settings =====
USER_PROMPT = 'A small red ceramic cup of hot coffee on a wooden table by a window, with delicate steam moving in soft morning light.'
USE_API_EXPANSION = True    # False means use USER_PROMPT as-is; no API request
API_MODEL = 'gpt-4o-mini'   # Replace if your API provider offers another model name
API_BASE_URL = 'https://api.openai.com/v1'           # Leave empty for OpenAI; for OpenAI-compatible gateways specify full base URL

SEED = 42
WIDTH, HEIGHT = 640, 352    # multiples of 16; safer than 832x480 on a T4
NUM_FRAMES = 17             # Wan needs 4*n+1 frames. 17 frames @ 8 FPS ≈ 2.1 s
FPS = 8
GEN_STEPS = 22             # 16 = faster, 28+ = slower / sometimes better
GUIDANCE_SCALE = 5.5

SEEDVR2_SHORT_EDGE = 704    # 352 → 704, approximately 2× spatial upscaling
SEEDVR2_MODEL = 'seedvr2_ema_3b_fp8_e4m3fn.safetensors'
assert NUM_FRAMES % 4 == 1
assert WIDTH % 16 == 0 and HEIGHT % 16 == 0
print('Output folder:', ROOT)


GPU: Tesla T4
VRAM (GiB): 14.56
Python: 3.13.16 | Torch: 2.14.1+cu130
Disk free (GiB): 235.4
Output folder: /content/video_onepass_demo


## 2. Install dependencies

Use the existing Colab PyTorch build (do not reinstall GPU torch).

In [5]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers>=0.35.1,<0.40', 'transformers>=4.49,<5', 'accelerate>=1.2',
    'huggingface_hub>=0.34,<1', 'safetensors', 'sentencepiece', 'protobuf',
    'imageio', 'imageio-ffmpeg', 'openai>=1.0,<3', 'ftfy'], check=True)
print('Installed generation and API dependencies.')
subprocess.run(['ffmpeg', '-version'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)


Installed generation and API dependencies.


CompletedProcess(args=['ffmpeg', '-version'], returncode=0, stdout=b'ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers\nbuilt with gcc 13 (Ubuntu 13.2.0-23ubuntu3)\nconfiguration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enab

## 3. Optional prompt expansion using your API

Your key is entered privately via `getpass` and **is not saved to the notebook**. Set `USE_API_EXPANSION = False` to skip this stage. Your API must support an OpenAI-compatible Chat Completions endpoint; otherwise, use the raw prompt.

In [6]:
PROMPT = USER_PROMPT
if USE_API_EXPANSION:
    from getpass import getpass
    from openai import OpenAI
    key = os.environ.get('OPENAI_API_KEY') or getpass('OpenAI-compatible API key: ')
    assert key.strip(), 'Empty API key. Either provide a key or set USE_API_EXPANSION=False.'
    client_kwargs = {'api_key': key}
    if API_BASE_URL.strip():
        client_kwargs['base_url'] = API_BASE_URL.strip()
    client = OpenAI(**client_kwargs)
    response = client.chat.completions.create(
        model=API_MODEL,
        messages=[
            {'role':'system','content':(
                'Expand a short request into one clear cinematic text-to-video prompt. '
                'Describe a single 2-second continuous shot, one primary subject, observable motion, '
                'camera movement, lighting, style, background. Avoid editing cuts, complicated actions, '
                'spoken dialogue, visible lettering, subtitles, logos, and instructions. '
                'Return ONLY the English video prompt (no commentary), under 120 words.')},
            {'role':'user','content':USER_PROMPT},
        ],
        temperature=0.5,
        max_tokens=220,
    )
    PROMPT = (response.choices[0].message.content or '').strip()
    assert PROMPT, 'API returned an empty prompt.'
    del client, response, key
print('Final generation prompt:')
print(PROMPT)
(ROOT / 'prompt.txt').write_text(PROMPT, encoding='utf-8')


OpenAI-compatible API key: ··········
Final generation prompt:
A small red ceramic cup filled with steaming hot coffee sits on a rustic wooden table, bathed in soft morning light streaming through a nearby window. The camera slowly zooms in, capturing the gentle rise of delicate steam curling upward from the cup, creating a warm and inviting atmosphere. The light casts a soft glow, highlighting the rich texture of the wood and the vibrant color of the cup. Outside the window, the blurred silhouette of trees sways gently in the morning breeze, enhancing the serene ambiance of the scene.


529

## 4. Wan2.1 1.3B — generate the source video

Model weights load from Hugging Face. We generate in a **separate Python process**, so all GPU/CPU memory can be released before loading SeedVR2. The pipeline uses FP16 on T4, a tiled VAE and sequential CPU offload (conservative, slower).

In [7]:
WAN_SCRIPT = r'''
import argparse, torch, gc
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

parser = argparse.ArgumentParser()
for arg, typ in [('prompt',str),('output',str),('width',int),('height',int),('frames',int),('steps',int),('fps',int),('seed',int)]:
    parser.add_argument('--'+arg, type=typ, required=True)
parser.add_argument('--guidance', type=float, default=5.5)
a = parser.parse_args()

assert torch.cuda.is_available(), 'CUDA/T4 required'
assert a.frames % 4 == 1, 'Wan2.1 expects 4*n+1 frames'
model_id = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
print('Downloading/loading', model_id, flush=True)
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.float16)
pipe.scheduler = UniPCMultistepScheduler(
    prediction_type='flow_prediction', use_flow_sigmas=True,
    num_train_timesteps=1000, flow_shift=3.0
)

# Avoid putting the full pipeline on T4 at once. Slow, but memory-conservative.
pipe.enable_sequential_cpu_offload()
if hasattr(pipe, 'enable_vae_tiling'):
    pipe.enable_vae_tiling()
elif hasattr(pipe.vae, 'enable_tiling'):
    pipe.vae.enable_tiling()

negative = 'blurry, low quality, noisy, static, flicker, jitter, visual artifacts, text, subtitles, watermark, cuts, deformed shapes'
print('Running video diffusion...', flush=True)
with torch.inference_mode():
    result = pipe(
        prompt=a.prompt, negative_prompt=negative, height=a.height, width=a.width,
        num_frames=a.frames, num_inference_steps=a.steps,
        guidance_scale=a.guidance, generator=torch.Generator(device='cpu').manual_seed(a.seed)
    )
frames = result.frames[0]
export_to_video(frames, a.output, fps=a.fps)
print('SOURCE_VIDEO_SAVED:', a.output, flush=True)
'''
script_path = ROOT / 'generate_wan.py'
script_path.write_text(WAN_SCRIPT, encoding='utf-8')
import ast
ast.parse(WAN_SCRIPT)
print('Saved runnable script:', script_path)


Saved runnable script: /content/video_onepass_demo/generate_wan.py


In [8]:
import subprocess
command = [
    sys.executable, str(ROOT / 'generate_wan.py'),
    '--prompt', PROMPT, '--output', str(SOURCE_MP4),
    '--width', str(WIDTH), '--height', str(HEIGHT),
    '--frames', str(NUM_FRAMES), '--fps', str(FPS),
    '--steps', str(GEN_STEPS), '--guidance', str(GUIDANCE_SCALE),
    '--seed', str(SEED)
]
if not SOURCE_MP4.is_file() or SOURCE_MP4.stat().st_size == 0:
    subprocess.run(command, check=True)
else:
    print('Source already exists; skipping Wan generation:', SOURCE_MP4)
assert SOURCE_MP4.is_file() and SOURCE_MP4.stat().st_size > 0, 'Wan video missing; check generation error above.'
print('Source video:', SOURCE_MP4, 'bytes:', SOURCE_MP4.stat().st_size)


Source video: /content/video_onepass_demo/source_wan.mp4 bytes: 42538


## 5. Preview the original MP4

In [9]:
from IPython.display import display, Video
display(Video(str(SOURCE_MP4), embed=True, html_attributes='controls loop'))

## 6. SeedVR2 3B FP8 — real video super-resolution

Using the [SeedVR2-CLI](https://github.com/delvilo/SeedVR2-CLI) community implementation, which documents Colab T4 support, memory scheduling, and chunking. **No ESRGAN fallback.**

If a future Colab update breaks TorchCodec/FFmpeg, consult the [project's Colab notebook](https://github.com/delvilo/SeedVR2-CLI/tree/main/colab) and rerun this step. T4 uses standard PyTorch SDPA, not FlashAttention 2.

In [10]:
import subprocess, os
SEEDVR2_REPO = pathlib.Path('/content/SeedVR2-CLI')
if not SEEDVR2_REPO.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/delvilo/SeedVR2-CLI.git', str(SEEDVR2_REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(SEEDVR2_REPO / 'requirements.txt')], check=True)
print('SeedVR2 dependencies installed. T4 attention backend: SDPA.')


SeedVR2 dependencies installed. T4 attention backend: SDPA.


In [11]:
# Choose a conservative 3B FP8, 5-frame batches, staged memory policy.
# --resolution is the shorter edge (704 means ~1280x704 for a 640x352 source).
seedvr_cmd = [
    sys.executable, str(SEEDVR2_REPO / 'inference_cli.py'),
    str(SOURCE_MP4),
    '--dit_model', SEEDVR2_MODEL,
    '--resolution', str(SEEDVR2_SHORT_EDGE),
    '--batch_size', '5',
    '--chunk_size', str(NUM_FRAMES),
    '--memory_policy', 'staged',
    '--attention_mode', 'sdpa',
    '--video_backend', 'ffmpeg',
    '--model_dir', str(ROOT / 'seedvr2_models'),
    '--output', str(UPSCALED_MP4),
]
print('SeedVR2 output:', UPSCALED_MP4)
if not UPSCALED_MP4.is_file() or UPSCALED_MP4.stat().st_size == 0:
    subprocess.run(seedvr_cmd, cwd=str(SEEDVR2_REPO), check=True)
else:
    print('Enhanced video already exists; skipping upscaling.')
assert UPSCALED_MP4.is_file() and UPSCALED_MP4.stat().st_size > 0,     'SeedVR2 failed to create an output. Check the traceback above.'
print('SeedVR2 SUCCESS:', UPSCALED_MP4, UPSCALED_MP4.stat().st_size, 'bytes')


SeedVR2 output: /content/video_onepass_demo/enhanced_seedvr2.mp4
SeedVR2 SUCCESS: /content/video_onepass_demo/enhanced_seedvr2.mp4 1247402 bytes


## 7. Verify outputs, preview, and download

In [12]:
import json, subprocess
from IPython.display import display, Video

def probe_video(path):
    p = subprocess.run(['ffprobe','-v','error','-select_streams','v:0',
        '-show_entries','stream=width,height,avg_frame_rate,nb_frames',
        '-of','json',str(path)],capture_output=True,text=True,check=True)
    return json.loads(p.stdout)['streams'][0]

print('Original:', probe_video(SOURCE_MP4))
print('SeedVR2 :', probe_video(UPSCALED_MP4))
display(Video(str(UPSCALED_MP4), embed=True, html_attributes='controls loop'))


Original: {'width': 640, 'height': 352, 'avg_frame_rate': '8/1', 'nb_frames': '17'}
SeedVR2 : {'width': 1280, 'height': 704, 'avg_frame_rate': '8/1', 'nb_frames': '17'}


In [13]:
# Download MP4s. If Colab output is very large, use the file sidebar instead.
from google.colab import files
files.download(str(UPSCALED_MP4))
# Optional: files.download(str(SOURCE_MP4))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Troubleshooting

**Generation GPU OOM / CPU RAM crash:** choose `WIDTH, HEIGHT = 512, 288`, `NUM_FRAMES = 9`, `GEN_STEPS = 16`, then remove `/content/video_onepass_demo/source_wan.mp4` and rerun generation. Sequential CPU offload avoids most VRAM issues, but cannot guarantee host RAM.

**SeedVR2 OOM:** reduce `SEEDVR2_SHORT_EDGE = 576`, `--batch_size` from `5` to `1`, and/or add `--blocks_to_swap 16` to `seedvr_cmd`; remove `enhanced_seedvr2.mp4` and rerun.

**API errors:** Set `USE_API_EXPANSION=False` to test video models independently. For third-party providers ensure `API_BASE_URL` and `API_MODEL` match that API.

**Free Colab disconnect:** You may rerun the notebook, but files in `/content/` are lost after a runtime reset. For persistence, copy source and output MP4 to Drive. Model caches in `/content/` must be downloaded again after reset.

**Quality:** A 17-frame, ~2-second clip is a resource-safe smoke test, not a production-quality video. After successful completion, raise to 33 frames and experiment with 25–30 generation steps. Increasing resolution and frames increases cost substantially.

**Implementation note:** This is a custom integration of the publicly documented Wan2.1 Diffusers API and the SeedVR2 CLI, not an officially tested end-to-end reference notebook. If a new version changes interfaces, the terminal traceback identifies which step failed.